# Small Files, Compaction and the Cost Curve

Companion notebook for the [Compaction, Clustering and File Sizing](https://codemia.io/courses/big_data_engines/bd_compaction_clustering_and_file_sizing) lesson on Codemia.

**What you will build**: a cost model for a table that holds the same bytes in a different number of files. You will price a
day long scan across four ingest configurations, watch per-file overhead overtake the bytes
themselves, pay for compaction and measure how fast it pays back, then sweep the compaction interval
to find the point where compacting more often starts costing more than it saves.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Concept recap

The small files problem is not about storage. The bytes are the same bytes, and object storage
charges you for them either way. It is about the fixed cost attached to each file, which shows up in
three places: the planner has to list and read metadata for every file, the reader has to issue a
separate request with its own round trip for every file, and the table format has to track every
file in a manifest that grows accordingly.

None of that is proportional to the data. It is proportional to the file count, which is set by how
your writers commit rather than by how much data you have. The model below puts numbers on all three
so you can see which one dominates at which scale.

## A cost model with the constants written down

In [ ]:
DAILY_GB = 240                 # one day of ingest
TARGET_MB = 128                # the file size compaction aims for
THREADS = 32                   # reader parallelism
THROUGHPUT_MB = 90             # sustained MB/s per thread from object storage
FILE_OVERHEAD_MS = 30          # per file: request round trip, footer read, decoder setup
PLANNER_FILES_PER_S = 20_000   # how fast the query planner chews through file metadata
MANIFEST_BYTES_PER_FILE = 1_500


def query_cost(files, data_gb, threads=THREADS):
    """Seconds for a scan of data_gb spread across the given number of files."""
    planning = files / PLANNER_FILES_PER_S
    per_file = files / threads * FILE_OVERHEAD_MS / 1000
    transfer = data_gb * 1024 / THROUGHPUT_MB / threads
    return {'total': planning + per_file + transfer, 'planning': planning,
            'per_file': per_file, 'transfer': transfer}


INGEST = [
    ('commit 60 s, 16 writers', 60, 16),
    ('commit 30 s, 16 writers', 30, 16),
    ('commit 10 s, 32 writers', 10, 32),
    ('commit 5 s, 64 writers', 5, 64),
]

print(f"{'ingest shape':>26}{'files/day':>11}{'avg file':>11}"
      f"{'plan s':>9}{'per-file s':>12}{'bytes s':>10}{'total s':>10}")
rows = []
for name, commit_s, writers in INGEST:
    files = 86_400 // commit_s * writers
    avg_mb = DAILY_GB * 1024 / files
    c = query_cost(files, DAILY_GB)
    rows.append((name, files, avg_mb, c))
    print(f'{name:>26}{files:>11,}{avg_mb:>9.2f} MB'
          f'{c["planning"]:>9.1f}{c["per_file"]:>12.1f}{c["transfer"]:>10.1f}'
          f'{c["total"]:>10.1f}')

compacted_files = int(np.ceil(DAILY_GB * 1024 / TARGET_MB))
c = query_cost(compacted_files, DAILY_GB)
rows.append((f'compacted to {TARGET_MB} MB', compacted_files, TARGET_MB, c))
print(f'{"compacted to 128 MB":>26}{compacted_files:>11,}{float(TARGET_MB):>9.2f} MB'
      f'{c["planning"]:>9.1f}{c["per_file"]:>12.1f}{c["transfer"]:>10.1f}{c["total"]:>10.1f}')

Every row reads the same two hundred and forty gigabytes. The bytes column is identical down the
table and it is the only honest work in the query; everything to the left of it is overhead that
exists because of a commit interval somebody chose for a different reason.

Notice where the crossover is. At one file every few seconds per writer, per-file overhead is a
fraction of the transfer time and nobody files a ticket. Shorten the commit and add writers, which
is exactly what you do when you are chasing ingest latency, and the overhead passes the transfer
time without any single change looking unreasonable.

In [ ]:
file_mb = np.logspace(np.log10(0.25), np.log10(1024), 60)
files = DAILY_GB * 1024 / file_mb
totals = [query_cost(f, DAILY_GB)['total'] for f in files]
floor = query_cost(0, DAILY_GB)['transfer']

plt.figure(figsize=(9.5, 5))
plt.plot(file_mb, totals, color='tab:blue', label='modelled scan time')
plt.axhline(floor, color='black', linestyle='--',
            label=f'floor: the bytes themselves ({floor:.0f} s)')
plt.axvline(TARGET_MB, color='tab:green', linestyle=':', label=f'{TARGET_MB} MB target')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('average file size (MB)')
plt.ylabel('seconds to scan one day')
plt.title('The same 240 GB, priced by how it was chopped up')
plt.legend()
plt.grid(alpha=0.3, which='both')
plt.show()

The curve is flat to the right of the target and steep to the left of it, which is the real reason
the hundred and twenty eight megabyte advice has survived so long. It is not that the number is
optimal. It is that being wrong on the high side costs you almost nothing, while being wrong on the
low side costs you everything, so the sensible target sits just inside the flat region.

## Paying for compaction, and getting it back

In [ ]:
CLUSTER_MB_S = THREADS * THROUGHPUT_MB          # what the whole reader fleet can move


def compaction_cost(data_gb):
    """A rewrite reads the data once and writes it once."""
    return data_gb * 1024 * 2 / CLUSTER_MB_S


worst = rows[3]                                  # commit every 5 s with 64 writers
best = rows[-1]                                  # the same data after compaction

pay = compaction_cost(DAILY_GB)
saving = worst[3]['total'] - best[3]['total']

print(f'before compaction  {worst[1]:>9,} files  {worst[3]["total"]:>8.1f} s per scan')
print(f'after compaction   {best[1]:>9,} files  {best[3]["total"]:>8.1f} s per scan')
print(f'saving per scan    {saving:>8.1f} s  ({worst[3]["total"] / best[3]["total"]:.1f}x faster)')
print()
print(f'compaction itself  {pay:>8.1f} s of cluster time, '
      f'{DAILY_GB * 2} GB of IO, {DAILY_GB} GB rewritten')
print(f'pays back after    {pay / saving:>8.2f} scans of that day')
print()
for queries in [1, 5, 20, 100]:
    before = worst[3]['total'] * queries
    after = best[3]['total'] * queries + pay
    print(f'{queries:>4} scans/day: {before:>8.0f} s uncompacted vs {after:>8.0f} s compacted '
          f'({before / after:>4.1f}x)')

The rewrite costs about a sixth of what a single scan of that day saves, so the first query after
compaction has already paid for it and everything after that is free.
This is why a compaction job that nobody asked for is usually the cheapest performance work
available on a lakehouse, and why the table formats all ship one.

It also tells you when not to bother. A partition written once and read once, a staging table that a
single job consumes and drops, a landing zone that exists for replay: those never reach the payback
point, and compacting them is pure cost.

## How often, and the write amplification nobody budgets for

In [ ]:
FANOUT = 10               # how many inputs a compaction round merges into one output
PARTITIONS = 64           # compaction runs per partition, so a window's data is divided by this
COMMIT_OVERHEAD_S = 15    # cluster time for one compaction run: planning, manifest rewrite, commit


def rewrite_amplification(window_gb, target_mb=TARGET_MB, fanout=FANOUT):
    """Data too small to reach the target on the first pass is rewritten again on the next.

    A tiered model: each round merges fanout inputs, so the number of times a byte is
    rewritten is the number of rounds it takes for a window to grow to target size.
    """
    window_mb = window_gb * 1024
    if window_mb >= target_mb:
        return 1.0
    rounds = np.ceil(np.log(target_mb / window_mb) / np.log(fanout)) + 1
    return float(rounds)


def daily_cost(interval_min, queries_per_day):
    runs = 1440 / interval_min
    window_gb = DAILY_GB / runs / PARTITIONS          # what one partition accumulates per window

    amp = rewrite_amplification(window_gb)
    rewrite_seconds = compaction_cost(DAILY_GB) * amp
    overhead_seconds = runs * COMMIT_OVERHEAD_S

    # A query sees compacted files plus whatever the current window has accumulated.
    tail_files = interval_min * 60 // 5 * 64          # the 5 s, 64 writer ingest shape
    files = compacted_files + tail_files
    read_seconds = query_cost(files, DAILY_GB)['total'] * queries_per_day

    return {'interval': interval_min, 'amp': amp, 'rewrite': rewrite_seconds,
            'overhead': overhead_seconds, 'read': read_seconds,
            'write': rewrite_seconds + overhead_seconds,
            'total': rewrite_seconds + overhead_seconds + read_seconds,
            'tail_files': tail_files}


QUERIES = 20
INTERVALS = [1, 2, 5, 10, 15, 30, 60, 120, 240, 480, 1440]

print(f"{'interval':>10}{'runs/day':>10}{'rewrites/byte':>15}{'rewrite s':>11}"
      f"{'run overhead s':>16}{'tail files':>12}{'query s':>10}{'total s':>10}")
sweep = [daily_cost(i, QUERIES) for i in INTERVALS]
for r in sweep:
    print(f'{r["interval"]:>8} m{int(1440 / r["interval"]):>10,}{r["amp"]:>15.1f}'
          f'{r["rewrite"]:>11.0f}{r["overhead"]:>16.0f}{r["tail_files"]:>12,}'
          f'{r["read"]:>10.0f}{r["total"]:>10.0f}')

best_interval = min(sweep, key=lambda r: r['total'])
print()
print(f'cheapest interval in this model: every {best_interval["interval"]} minutes, '
      f'{best_interval["total"]:.0f} s of cluster time per day')

In [ ]:
intervals = [r['interval'] for r in sweep]
plt.figure(figsize=(9.5, 5))
plt.plot(intervals, [r['rewrite'] for r in sweep], marker='o', label='rewritten bytes')
plt.plot(intervals, [r['overhead'] for r in sweep], marker='o', label='per-run fixed cost')
plt.plot(intervals, [r['read'] for r in sweep], marker='o', label=f'queries ({QUERIES}/day)')
plt.plot(intervals, [r['total'] for r in sweep], marker='o', color='black', label='total')
plt.axvline(best_interval['interval'], color='tab:green', linestyle=':',
            label=f'cheapest: every {best_interval["interval"]} min')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('compaction interval (minutes)')
plt.ylabel('cluster seconds per day')
plt.title('Compact too often and you rewrite the same bytes; too rarely and every query pays')
plt.legend()
plt.grid(alpha=0.3, which='both')
plt.show()

The left arm of the curve is the one that surprises people, and it has two ingredients. The first is
the fixed cost of a compaction run: planning, rewriting manifests and winning an optimistic
concurrency commit cost roughly the same whether the run moves one gigabyte or a hundred, and
running it every minute means paying that cost fourteen hundred times a day. The second is rewrite
amplification. A single partition's one minute window holds a couple of megabytes, nowhere near the
target, so the first pass cannot produce a full file and a later round picks the output up again.
That is the `rewrites per byte` column, and you pay it in cluster time and in object storage write
requests, which are priced per request rather than per byte.

The partition count is what makes this bite. Dividing a day of ingest by a few dozen partitions
leaves each partition with far less than a target sized file per window, which is why heavily
partitioned tables are the ones with chronic small file problems and why adding a partition column
is never free.

The right arm is the familiar one, where the uncompacted tail grows until queries are back to
reading hundreds of thousands of files.

The flat middle is wide, which is the practical lesson. Everything from a quarter of an hour to two
hours lands within a quarter of the cheapest point in this model, so the choice inside that range
should be made on how fresh the compacted view needs to be rather than on cost. Only the extremes
are mistakes.

## The second file count: deletes

In [ ]:
DELETE_FILES_PER_COMMIT = 8        # position deletes land next to the data they shadow


def mor_query_cost(commits_since_compaction, data_gb=DAILY_GB):
    delete_files = commits_since_compaction * DELETE_FILES_PER_COMMIT
    base = query_cost(compacted_files, data_gb)['total']
    deletes = query_cost(delete_files, 0)['total']
    return base, deletes, base + deletes


print(f"{'commits since compaction':>26}{'delete files':>14}{'base s':>9}"
      f"{'deletes s':>11}{'total s':>9}")
for commits in [0, 60, 360, 1_440, 8_640]:
    base, deletes, total = mor_query_cost(commits)
    print(f'{commits:>26,}{commits * DELETE_FILES_PER_COMMIT:>14,}'
          f'{base:>9.1f}{deletes:>11.1f}{total:>9.1f}')

Merge on read moves the cost of an update from the writer to every subsequent reader, and the
currency it pays in is file count again. A table with a tidy set of data files and a day of
accumulated position deletes reads badly for a reason that no amount of data file compaction fixes,
because the delete files are a separate population with their own manifest and their own requests.

This is why Iceberg and Delta expose two different maintenance operations, and why a table under
frequent updates needs both: rewrite the data files for size, and rewrite or apply the delete files
for count. A maintenance job that only does the first leaves half the problem in place.

## What to take away

- Per-file overhead is independent of data volume, so the small files problem is created by the
  commit interval and the writer count, not by the size of the table.
- Scan time has a floor set by bytes and throughput. File sizing is the difference between sitting
  on that floor and sitting an order of magnitude above it.
- The cost curve is asymmetric around the target. Oversized files are nearly free; undersized files
  are not. Aim slightly high.
- Compaction usually pays for itself within a scan or two, which makes it the cheapest tuning
  available, except on tables nobody scans twice.
- Compacting too frequently rewrites the same bytes several times. The cheap region is wide, so pick
  the interval for freshness and leave it alone.
- Count your delete files as well as your data files. Merge on read hands the reader a second file
  population, and only delete compaction removes it.